## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../src')

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Load Models

In [ ]:
# Load pre-trained models
print("Loading YOLOv8 detection model...")
model_det = YOLO('yolov8n.pt')

print("Loading YOLOv8-Pose model...")
model_pose = YOLO('yolov8n-pose.pt')

print("✅ Models loaded successfully!")

## 3. Test on Sample Image

In [ ]:
# Find a sample image
frames_dir = Path('../data/frames')
sample_images = list(frames_dir.rglob('*.jpg'))

if not sample_images:
    print("⚠️ No images found. Run frame extraction first.")
else:
    sample_img = str(sample_images[0])
    print(f"Testing on: {Path(sample_img).name}")
    
    # Display original
    img = cv2.imread(sample_img)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(img_rgb)
    plt.title('Original Image')
    plt.axis('off')
    plt.show()

## 4. Run Detection

In [ ]:
# Run detection
results_det = model_det.predict(sample_img, conf=0.25, verbose=False)

# Display results
annotated_det = results_det[0].plot()
annotated_det_rgb = cv2.cvtColor(annotated_det, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(annotated_det_rgb)
plt.title(f'Detection Results - {len(results_det[0].boxes)} objects detected')
plt.axis('off')
plt.show()

# Print detection details
print(f"\nDetections: {len(results_det[0].boxes)}")
for i, box in enumerate(results_det[0].boxes):
    cls = int(box.cls[0])
    conf = float(box.conf[0])
    print(f"  {i+1}. Class: {results_det[0].names[cls]}, Confidence: {conf:.3f}")

## 5. Run Pose Estimation

In [ ]:
# Run pose estimation
results_pose = model_pose.predict(sample_img, conf=0.25, verbose=False)

# Display results
annotated_pose = results_pose[0].plot()
annotated_pose_rgb = cv2.cvtColor(annotated_pose, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(annotated_pose_rgb)
plt.title(f'Pose Estimation - {len(results_pose[0].keypoints)} persons detected')
plt.axis('off')
plt.show()

# Print keypoint details
if hasattr(results_pose[0], 'keypoints') and len(results_pose[0].keypoints) > 0:
    print(f"\nPersons detected: {len(results_pose[0].keypoints)}")
    print(f"Keypoints per person: 17 (COCO format)")
    
    # Show average confidence
    all_confs = results_pose[0].keypoints.conf.cpu().numpy().flatten()
    print(f"Average keypoint confidence: {np.mean(all_confs):.3f}")

## 6. Process Multiple Frames

In [ ]:
# Process first 5 frames from each video
num_samples = min(5, len(sample_images))

fig, axes = plt.subplots(num_samples, 2, figsize=(15, 5*num_samples))
if num_samples == 1:
    axes = axes.reshape(1, -1)

for idx in range(num_samples):
    img_path = str(sample_images[idx])
    
    # Detection
    res_det = model_det.predict(img_path, conf=0.25, verbose=False)
    det_img = cv2.cvtColor(res_det[0].plot(), cv2.COLOR_BGR2RGB)
    
    # Pose
    res_pose = model_pose.predict(img_path, conf=0.25, verbose=False)
    pose_img = cv2.cvtColor(res_pose[0].plot(), cv2.COLOR_BGR2RGB)
    
    # Display
    axes[idx, 0].imshow(det_img)
    axes[idx, 0].set_title(f'Frame {idx+1} - Detection ({len(res_det[0].boxes)} objects)')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(pose_img)
    axes[idx, 1].set_title(f'Frame {idx+1} - Pose ({len(res_pose[0].keypoints)} persons)')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

## 7. Video Processing (Single Video)

In [ ]:
# Find first video
videos_dir = Path('../data/raw_videos')
videos = list(videos_dir.glob('*.mp4')) + list(videos_dir.glob('*.avi'))

if not videos:
    print("⚠️ No videos found. Download videos first.")
else:
    video_path = str(videos[0])
    print(f"Processing: {Path(video_path).name}")
    
    # Run detection on video
    output_dir = Path('../outputs/test_notebook')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("\nRunning detection...")
    results = model_det.predict(
        video_path,
        conf=0.25,
        save=True,
        project=str(output_dir),
        name='detection',
        exist_ok=True
    )
    
    print(f"✅ Detection complete! Output saved to: {output_dir / 'detection'}")
    
    print("\nRunning pose estimation...")
    results_pose = model_pose.predict(
        video_path,
        conf=0.25,
        save=True,
        project=str(output_dir),
        name='pose',
        exist_ok=True
    )
    
    print(f"✅ Pose estimation complete! Output saved to: {output_dir / 'pose'}")

## 8. Performance Statistics

In [ ]:
# Analyze results from multiple frames
if sample_images:
    num_test = min(20, len(sample_images))
    
    det_counts = []
    pose_counts = []
    confidences = []
    
    print(f"Analyzing {num_test} frames...\n")
    
    for img_path in sample_images[:num_test]:
        # Detection
        res_det = model_det.predict(str(img_path), conf=0.25, verbose=False)
        det_counts.append(len(res_det[0].boxes))
        
        # Pose
        res_pose = model_pose.predict(str(img_path), conf=0.25, verbose=False)
        num_persons = len(res_pose[0].keypoints) if hasattr(res_pose[0], 'keypoints') else 0
        pose_counts.append(num_persons)
        
        if num_persons > 0:
            confs = res_pose[0].keypoints.conf.cpu().numpy().flatten()
            confidences.extend(confs.tolist())
    
    # Statistics
    print("📊 Statistics:")
    print(f"   Avg detections per frame: {np.mean(det_counts):.2f}")
    print(f"   Avg persons per frame: {np.mean(pose_counts):.2f}")
    print(f"   Avg keypoint confidence: {np.mean(confidences):.3f}")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].bar(['Detections', 'Persons'], [np.mean(det_counts), np.mean(pose_counts)])
    axes[0].set_ylabel('Count')
    axes[0].set_title('Average Counts per Frame')
    axes[0].grid(alpha=0.3)
    
    axes[1].hist(det_counts, bins=10, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Detections per Frame')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Detection Distribution')
    axes[1].grid(alpha=0.3)
    
    axes[2].hist(confidences, bins=30, edgecolor='black', alpha=0.7)
    axes[2].set_xlabel('Keypoint Confidence')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Keypoint Confidence Distribution')
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 9. Next Steps

After exploring in this notebook:

1. **Process all videos:**
   ```bash
   python src/03_inference_detection.py --source data/raw_videos --output outputs/detections
   python src/04_pose_inference.py --source data/raw_videos --output outputs/poses
   ```

2. **Evaluate performance:**
   ```bash
   python src/05_evaluation.py --pose-keypoints outputs/poses/[video]_pose.json
   ```

3. **Collect screenshots** for report from `outputs/` directories

4. **Complete report** in `report/report.md`